# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  

**`Student Name`:**  Tisha Bhavsar

**`Roll Number`:**  U20230133

**`GitHub Branch`:** tisha_U20230xxx  

# Imports and Setup

In [246]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler

# Load Datasets

In [247]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

#### Exploring the Dataset

In [248]:
news_df.shape

(209527, 6)

In [249]:
news_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               209527 non-null  str  
 1   headline           209521 non-null  str  
 2   category           209527 non-null  str  
 3   short_description  189815 non-null  str  
 4   authors            172109 non-null  str  
 5   date               209527 non-null  str  
dtypes: str(6)
memory usage: 9.6 MB


#### Checking News Categories


In [250]:
print(f"Total unique categories: {news_df['category'].nunique()}")
print(news_df['category'].unique())

Total unique categories: 42
<StringArray>
[     'U.S. NEWS',         'COMEDY',      'PARENTING',     'WORLD NEWS',
 'CULTURE & ARTS',           'TECH',         'SPORTS',  'ENTERTAINMENT',
       'POLITICS',     'WEIRD NEWS',    'ENVIRONMENT',      'EDUCATION',
          'CRIME',        'SCIENCE',       'WELLNESS',       'BUSINESS',
 'STYLE & BEAUTY',   'FOOD & DRINK',          'MEDIA',   'QUEER VOICES',
  'HOME & LIVING',          'WOMEN',   'BLACK VOICES',         'TRAVEL',
          'MONEY',       'RELIGION',  'LATINO VOICES',         'IMPACT',
       'WEDDINGS',        'COLLEGE',        'PARENTS', 'ARTS & CULTURE',
          'STYLE',          'GREEN',          'TASTE', 'HEALTHY LIVING',
  'THE WORLDPOST',      'GOOD NEWS',      'WORLDPOST',          'FIFTY',
           'ARTS',        'DIVORCE']
Length: 42, dtype: str


**Inference**: We only need 4 categories: ENTERTAINMENT, EDUCATION, TECH, CRIME

In [251]:
required_categories = ['ENTERTAINMENT', 'EDUCATION', 'TECH', 'CRIME']
news_filtered = news_df[news_df['category'].isin(required_categories)].copy()

print(f"Original articles: {len(news_df)}")
print(f"Filtered articles: {len(news_filtered)}")
print(f"Filtered category distribution:")
print(news_filtered['category'].value_counts())

Original articles: 209527
Filtered articles: 24042
Filtered category distribution:
category
ENTERTAINMENT    17362
CRIME             3562
TECH              2104
EDUCATION         1014
Name: count, dtype: int64


#### Handling Missing Values

In [252]:
missing_news = news_df.isnull().sum()
print("Missing values in news dataset:")
print(missing_news)
print(f"\nTotal rows: {len(news_df)}")

Missing values in news dataset:
link                     0
headline                 6
category                 0
short_description    19712
authors              37418
date                     0
dtype: int64

Total rows: 209527


**Observation:** Most missing values are in `short_description` and `authors` columns. These columns are not needed for our recommendation system, so we can ignore them. For the missing values in the headline column, we can drop them as the number of rows are enough.


In [253]:
print(f"Missing headlines: {news_filtered['headline'].isnull().sum()}")
news_filtered = news_filtered.dropna(subset=['headline'])
print("After dropping missing headlines: {len(news_filtered)}")

Missing headlines: 0
After dropping missing headlines: {len(news_filtered)}


In [254]:
news_filtered = news_filtered[['category', 'headline', 'link']].reset_index(drop=True)
print(f"Final news dataset shape: {news_filtered.shape}")
news_filtered.head()

Final news dataset shape: (24042, 3)


,category,headline,link
0,TECH,Twitch Bans Gambling Sites After Streamer Scam...,https://www.huffpost.com/entry/twitch-streamer...
1,ENTERTAINMENT,Golden Globes Returning To NBC In January Afte...,https://www.huffpost.com/entry/golden-globes-r...
2,ENTERTAINMENT,James Cameron Says He 'Clashed' With Studio Be...,https://www.huffpost.com/entry/james-cameron-f...
3,ENTERTAINMENT,Amazon Greenlights 'Blade Runner 2099' Limited...,https://www.huffpost.com/entry/blade-runner-20...
4,ENTERTAINMENT,'The Phantom Of The Opera' To Close On Broadwa...,https://www.huffpost.com/entry/the-phantom-of-...


#### Exploring User Datasets

In [255]:
train_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      2000 non-null   str    
 1   age                          1302 non-null   float64
 2   income                       2000 non-null   int64  
 3   clicks                       2000 non-null   int64  
 4   purchase_amount              2000 non-null   float64
 5   session_duration             2000 non-null   float64
 6   content_variety              2000 non-null   float64
 7   engagement_score             2000 non-null   float64
 8   num_transactions             2000 non-null   int64  
 9   avg_monthly_spend            2000 non-null   float64
 10  avg_cart_value               2000 non-null   float64
 11  browsing_depth               2000 non-null   int64  
 12  revisit_rate                 2000 non-null   float64
 13  scroll_activity              

In [256]:
train_users.shape

(2000, 33)

In [257]:
test_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      2000 non-null   str    
 1   age                          1321 non-null   float64
 2   income                       2000 non-null   int64  
 3   clicks                       2000 non-null   int64  
 4   purchase_amount              2000 non-null   float64
 5   session_duration             2000 non-null   float64
 6   content_variety              2000 non-null   float64
 7   engagement_score             2000 non-null   float64
 8   num_transactions             2000 non-null   int64  
 9   avg_monthly_spend            2000 non-null   float64
 10  avg_cart_value               2000 non-null   float64
 11  browsing_depth               2000 non-null   int64  
 12  revisit_rate                 2000 non-null   float64
 13  scroll_activity              

In [258]:
test_users.shape

(2000, 32)

#### Checking for Missing Values

In [259]:
print("Missing values in train_users:")
print(train_users.isnull().sum())
print("\nMissing values in test_users:")
print(test_users.isnull().sum())

Missing values in train_users:
user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count    

**Handle Missing Values in Age Column**: We'll impute missing age values using the median age strategy.

In [260]:
train_users['age'] = train_users['age'].fillna(train_users['age'].median())
test_users['age'] = test_users['age'].fillna(test_users['age'].median())

print("✓ Missing values handled")
print(f"Train missing age: {train_users['age'].isnull().sum()}")
print(f"Test missing age: {test_users['age'].isnull().sum()}")

✓ Missing values handled
Train missing age: 0
Test missing age: 0


#### Feature encoding
We have non-numeric columns: `browser_version`, `region_code`, and `subscriber` (boolean)

In [261]:
from sklearn.preprocessing import LabelEncoder

# Drop user_id (not a feature)
X_train = train_users.drop(['user_id', 'label'], axis=1)
y_train = train_users['label']

X_test = test_users.drop(['user_id'], axis=1)

cat_cols = ['browser_version', 'region_code', 'subscriber']

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)])
    le.fit(combined)
    
    # Transform both
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (2000, 31)
Test shape: (2000, 31)


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [262]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Train set: {X_tr.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"\nTrain label distribution:\n{y_tr.value_counts()}")
print(f"\nValidation label distribution:\n{y_val.value_counts()}")

Train set: 1600 samples
Validation set: 400 samples

Train label distribution:
label
user_2    570
user_1    565
user_3    465
Name: count, dtype: int64

Validation label distribution:
label
user_2    142
user_1    142
user_3    116
Name: count, dtype: int64


In [263]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {acc:.4f}\n")
print("Classification Report:")
print(classification_report(y_val, y_pred))

Validation Accuracy: 0.8975

Classification Report:
              precision    recall  f1-score   support

      user_1       0.89      0.87      0.88       142
      user_2       0.97      0.87      0.92       142
      user_3       0.84      0.97      0.90       116

    accuracy                           0.90       400
   macro avg       0.90      0.90      0.90       400
weighted avg       0.90      0.90      0.90       400



In [264]:
# Predict user categories for test set
test_predictions = clf.predict(X_test)
test_users['predicted_label'] = test_predictions

print("Test set predictions generated")
print(f"\nPredicted distribution:\n{test_users['predicted_label'].value_counts()}")

Test set predictions generated

Predicted distribution:
predicted_label
user_2    703
user_1    675
user_3    622
Name: count, dtype: int64


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
